# Stanford CS25: How I Learned to Stop Worrying and Love the Transformer
### An Interactive, Executable Deep Dive

This notebook serves as an expert-level, interactive companion to the Stanford CS25 lecture on the Transformer architecture. We will deconstruct the core concepts, from the historical context that motivated its creation to the mathematical underpinnings and practical code implementations. Our goal is to build a deep, intuitive, and technical understanding of one of the most influential architectures in modern AI.

## 📚 Section 1: Learning Overview & Prerequisites

### Executive Summary of the Lecture

The lecture traces the evolution of Natural Language Processing (NLP) models, highlighting the journey from complex, multi-component systems to the unified and powerful Transformer architecture. It begins with the ambitious goals of the 1955 Dartmouth Conference, which envisioned a single machine simulating all of human intelligence, a goal that proved elusive for decades.

The speaker contrasts this with the reality of NLP in the 2000s and early 2010s, characterized by fragmented, specialized, and highly-engineered pipelines (e.g., statistical machine translation). The introduction of neural networks, particularly Recurrent Neural Networks (RNNs) and LSTMs, began a process of consolidation. These models, exemplified by Google's Neural Machine Translation system, replaced complex pipelines with a more homogeneous neural architecture.

However, LSTMs had a fundamental flaw: **sequentiality**. They processed information token-by-token, creating a computational bottleneck and making it difficult to capture long-range dependencies. This limitation spurred research into parallelizable architectures like convolutions and, ultimately, attention.

The **Transformer**, introduced in "Attention Is All You Need," marked a paradigm shift. It completely abandoned recurrence and convolutions in favor of **self-attention**. This mechanism allows any token in a sequence to directly interact with any other token, capturing dependencies regardless of distance and enabling massive parallelization during training.

Key components of the Transformer are detailed:
1.  **Scaled Dot-Product Attention**: The core mechanism for calculating how much focus one token should place on others.
2.  **Multi-Head Attention**: An enhancement that allows the model to focus on different types of information (e.g., syntactic vs. semantic) from different representation subspaces in parallel.
3.  **Positional Encodings**: A necessary addition to inject information about token order, since the self-attention mechanism itself is permutation-invariant.
4.  **Encoder-Decoder Stacks**: The overall architecture comprising layers of attention and feed-forward networks with residual connections and layer normalization.

The lecture concludes by discussing the evolution of the Transformer, including improvements like Rotary Positional Encodings and efficient attention mechanisms like FlashAttention, and its profound impact, leading to the era of Large Language Models (LLMs) and bringing us closer to the original vision of the Dartmouth Conference.

### All Prerequisite Topics

- **Linear Algebra**: Vectors, matrices, dot product, matrix multiplication.
- **Calculus**: Gradients, chain rule (for understanding backpropagation).
- **Probability & Statistics**: Basic probability theory, probability distributions.
- **Machine Learning**: Neural Networks, hidden layers, activation functions (ReLU, Softmax), embeddings, loss functions, gradient descent, backpropagation.
- **Python Programming**: Proficiency with Python syntax and data structures.
- **NumPy & PyTorch**: Experience with tensor operations in these libraries is essential.

### Topics Covered in this Notebook

1.  **Mathematical Foundations**
    - The Dot Product as a Similarity Measure
    - The Softmax Function for Normalization
    - Matrix Operations for Attention
2.  **Prerequisite Concepts: The Problem with Sequentiality**
    - Implementing a simple RNN to understand the bottleneck
    - Visualizing the sequential data flow
3.  **Core Research Content: The Transformer Architecture**
    - **Scaled Dot-Product Attention**: The core building block.
    - **Multi-Head Attention**: Focusing on different representation subspaces.
    - **Positional Encodings**: Reintroducing sequence order.
    - **Encoder & Decoder Layers**: Assembling the blocks.
    - **The Full Transformer**: A complete model for a toy task.
4.  **Experimental Analysis & Intuition Building**
    - The effect of the scaling factor in attention.
    - The necessity of positional encodings.
    - Analyzing what different attention heads learn.
5.  **Advanced Topics & Research Extensions**
    - Efficient Attention: The idea behind FlashAttention.
    - Improved Positional Encodings: The intuition of Rotary Embeddings (RoPE).
    - Faster Inference: The concept of Speculative Decoding.

### 🧠 Interactive Prerequisite Knowledge Checker

Test your foundational knowledge before diving in. A score of 3/3 is recommended.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

questions = [
    {
        'question': "What does the dot product of two unit vectors measure?",
        'options': ['The distance between them', 'The cosine of the angle between them', 'The sum of their elements', 'The perpendicular vector'],
        'correct': 1
    },
    {
        'question': "What is the primary purpose of the Softmax function in a neural network classification layer?",
        'options': ['To normalize inputs to have zero mean', 'To prevent overfitting', 'To convert raw scores (logits) into a probability distribution', 'To speed up training'],
        'correct': 2
    },
    {
        'question': "In a simple RNN, how is information from the first token 'The' passed to the third token 'cat' in the sequence 'The quick brown cat'?",
        'options': ['Through a direct connection', 'It is not passed at all', 'It is summarized in a hidden state that is updated sequentially at each step', 'Via a global attention mechanism'],
        'correct': 2
    }
]

quiz_widgets = []
for i, q in enumerate(questions):
    radio_buttons = widgets.RadioButtons(
        options=q['options'],
        description=f"Q{i+1}: {q['question']}",
        disabled=False,
        style={'description_width': 'initial'}
    )
    quiz_widgets.append(radio_buttons)
    display(radio_buttons)

button = widgets.Button(description="Check Answers")
output = widgets.Output()

def check_answers(b):
    score = 0
    with output:
        output.clear_output()
        for i, q in enumerate(questions):
            if quiz_widgets[i].index == q['correct']:
                score += 1
        print(f"You scored {score}/{len(questions)}. {'Great job!' if score == 3 else 'You may want to review the prerequisites.'}")

button.on_click(check_answers)
display(button, output)

## 🔢 Section 2: Mathematical Foundations

### The Dot Product as a Similarity Measure

At the heart of attention is the concept of similarity. How do we determine if two words (represented as vectors) are related? The dot product provides a simple yet powerful way to measure this.

#### Mathematical Definition

For two vectors $\mathbf{a}$ and $\mathbf{b}$, their dot product is defined as:
$$ \mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{n} a_i b_i = \|\mathbf{a}\| \|\mathbf{b}\| \cos(\theta) $$
Where $\|\mathbf{a}\|$ is the magnitude (or length) of vector $\mathbf{a}$, and $\theta$ is the angle between the two vectors.

#### Intuitive Understanding

If we assume the vectors are normalized to have a magnitude of 1 (unit vectors), the formula simplifies to $\mathbf{a} \cdot \mathbf{b} = \cos(\theta)$.
- When vectors point in the **same direction**, $\theta = 0$, so $\cos(\theta) = 1$. The dot product is high.
- When vectors are **orthogonal** (unrelated), $\theta = 90^\circ$, so $\cos(\theta) = 0$. The dot product is zero.
- When vectors point in **opposite directions**, $\theta = 180^\circ$, so $\cos(\theta) = -1$. The dot product is low.

In the Transformer, we use the dot product between a "Query" vector (representing the current word's question, e.g., "Who is performing an action?") and a "Key" vector (representing another word's potential answer, e.g., "I am a verb.") to get a raw **attention score**. A high score means the key is highly relevant to the query.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_vectors(vecs, colors, title):
    plt.figure(figsize=(6, 6))
    plt.axhline(0, color='grey', lw=0.5)
    plt.axvline(0, color='grey', lw=0.5)
    for vec, color in zip(vecs, colors):
        plt.quiver(0, 0, vec[0], vec[1], angles='xy', scale_units='xy', scale=1, color=color, label=f'{vec}')
    max_val = np.max(np.abs(vecs)) * 1.2
    plt.xlim(-max_val, max_val)
    plt.ylim(-max_val, max_val)
    plt.grid(True)
    plt.title(title)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.show()

@widgets.interact(
    a1=widgets.FloatSlider(value=1.0, min=-2.0, max=2.0, step=0.1, description='a.x'),
    a2=widgets.FloatSlider(value=1.5, min=-2.0, max=2.0, step=0.1, description='a.y'),
    b1=widgets.FloatSlider(value=1.2, min=-2.0, max=2.0, step=0.1, description='b.x'),
    b2=widgets.FloatSlider(value=0.5, min=-2.0, max=2.0, step=0.1, description='b.y'),
)
def interactive_dot_product_explorer(a1, a2, b1, b2):
    a = np.array([a1, a2])
    b = np.array([b1, b2])
    
    dot_product = np.dot(a, b)
    
    # Normalize for color mapping
    norm_a = a / (np.linalg.norm(a) + 1e-6)
    norm_b = b / (np.linalg.norm(b) + 1e-6)
    cosine_sim = np.dot(norm_a, norm_b)
    
    color_val = (cosine_sim + 1) / 2 # Map [-1, 1] to [0, 1]
    title_color = plt.cm.coolwarm(color_val)

    title = f'Vectors a & b\nDot Product: {dot_product:.2f} | Cosine Similarity: {cosine_sim:.2f}'
    plot_vectors([a, b], ['red', 'blue'], title)
    print("Adjust the vector components. Observe how the dot product and cosine similarity change.")
    print("When vectors are aligned, similarity is high (warm color). When they are opposed, it's low (cool color).")

### The Softmax Function

After calculating dot products between a query and all keys in a sequence, we get a set of raw attention scores. Some might be high, some low, some negative. To turn these into a useful distribution of weights (i.e., percentages of focus), we use the softmax function.

#### Mathematical Definition

Given a vector of scores $\mathbf{z} = (z_1, z_2, ..., z_k)$, the softmax function computes a probability vector $\mathbf{p}$ where each element $p_i$ is:
$$ p_i = \text{softmax}(\mathbf{z})_i = \frac{e^{z_i}}{\sum_{j=1}^{k} e^{z_j}} $$

#### Properties & Intuition
1.  **Normalization**: The output values are all between 0 and 1, and they sum to 1, just like a probability distribution.
2.  **Exponentiation**: The use of $e^x$ makes all scores positive and exaggerates differences. A score of 4 becomes much larger relative to a score of 2 than it was initially ($e^4 \approx 54.6$ vs $e^2 \approx 7.4$). This helps the model focus more confidently on the most relevant tokens.
3.  **Numerical Stability**: In practice, if scores $z_i$ are very large, $e^{z_i}$ can overflow. A common trick is to subtract the maximum score from all scores before exponentiating: $z'_i = z_i - \max(\mathbf{z})$. This doesn't change the output but prevents overflow. This is known as the **log-sum-exp trick**.

In [ ]:
def softmax_educational(z):
    """Educational implementation of softmax for clarity."""
    # For numerical stability, subtract the max value
    # This prevents overflow when scores are large
    stable_z = z - np.max(z)
    print(f"Original scores: {z}")
    print(f"Stabilized scores (z - max(z)): {stable_z}")
    
    # Exponentiate the stabilized scores
    exps = np.exp(stable_z)
    print(f"Exponentiated scores: {np.round(exps, 2)}")
    
    # Normalize to get probabilities
    sum_of_exps = np.sum(exps)
    print(f"Sum of exponentiated scores: {sum_of_exps:.2f}")
    
    probabilities = exps / sum_of_exps
    print(f"Final probabilities: {np.round(probabilities, 2)}")
    print(f"Sum of probabilities: {np.sum(probabilities):.2f}")
    return probabilities

def softmax_optimized(z):
    """Optimized implementation of softmax."""
    z = z - np.max(z, axis=-1, keepdims=True)
    exps = np.exp(z)
    return exps / np.sum(exps, axis=-1, keepdims=True)

# Let's test the educational version
print("--- Educational Softmax Example ---")
scores = np.array([1.0, 3.0, 0.5, 2.0])
_ = softmax_educational(scores)

# Interactive Visualization
@widgets.interact(
    s1=widgets.FloatSlider(value=1.0, min=-5.0, max=5.0, step=0.5, description='Score 1'),
    s2=widgets.FloatSlider(value=3.0, min=-5.0, max=5.0, step=0.5, description='Score 2'),
    s3=widgets.FloatSlider(value=0.5, min=-5.0, max=5.0, step=0.5, description='Score 3'),
    temperature=widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='Temperature')
)
def interactive_softmax_explorer(s1, s2, s3, temperature):
    scores = np.array([s1, s2, s3])
    
    # Temperature is a a hyperparameter that scales the logits before softmax.
    # High temp -> softer probabilities. Low temp -> harder, more confident probabilities.
    scaled_scores = scores / temperature
    
    probs = softmax_optimized(scaled_scores)
    
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.bar(['Score 1', 'Score 2', 'Score 3'], scores, color='skyblue')
    plt.title('Raw Scores (Logits)')
    plt.ylim(min(scores.min()-1, -5), max(scores.max()+1, 5))
    
    plt.subplot(1, 2, 2)
    plt.bar(['Prob 1', 'Prob 2', 'Prob 3'], probs, color='salmon')
    plt.title(f'Softmax Probabilities (Temp: {temperature:.1f})')
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()
    print("Note how a higher temperature makes the distribution 'softer' (more uniform), while a lower temperature makes it 'harder' (more spiky).")

## ⚙️ Section 3: Prerequisite Algorithms - The Problem with Sequentiality

### Recurrent Neural Networks (RNNs)

Before the Transformer, RNNs (and their more powerful variants, LSTMs and GRUs) were the state-of-the-art for sequence modeling. The lecture highlights their key weakness: sequential processing. To understand why this is a problem, let's implement a simple RNN cell from scratch.

An RNN processes a sequence one token at a time. At each step $t$, it takes the input for that step, $x_t$, and the hidden state from the previous step, $h_{t-1}$, to produce a new hidden state, $h_t$.

$$ h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h) $$

This hidden state $h_t$ is a compressed summary of all information seen up to that point. This creates two major problems:

1.  **Information Bottleneck**: The entire history of a potentially long sentence must be squeezed into a single fixed-size vector ($h_t$). This can lead to information loss, especially for long-range dependencies.
2.  **Lack of Parallelism**: To compute $h_t$, you *must* have already computed $h_{t-1}$. This dependency prevents parallel computation across the sequence dimension, making training on long sequences very slow.

In [ ]:
import torch
import torch.nn as nn

class SimpleRNN(nn.Module):
    """A simple RNN implementation from scratch for educational purposes."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.W_xh = nn.Parameter(torch.randn(hidden_size, input_size))
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size))
        self.b_h = nn.Parameter(torch.zeros(hidden_size))
        
    def forward(self, x):
        # x shape: (seq_len, batch_size, input_size)
        seq_len, batch_size, _ = x.shape
        
        # Initialize hidden state
        h_prev = torch.zeros(batch_size, self.hidden_size)
        
        # Store hidden states for visualization
        hidden_states = []
        
        print("--- RNN Sequential Processing --- ")
        # This loop demonstrates the sequential dependency
        for t in range(seq_len):
            x_t = x[t]
            
            # The core RNN calculation
            h_next = torch.tanh(
                x_t @ self.W_xh.T + 
                h_prev @ self.W_hh.T + 
                self.b_h
            )
            print(f"Step {t+1}: Processed token {t+1}. Hidden state updated.")
            
            h_prev = h_next
            hidden_states.append(h_next)
        
        print("\nThis sequential loop is the bottleneck! We can't calculate step 4 without step 3.")
        return torch.stack(hidden_states)

# --- Parameters ---
input_size = 10
hidden_size = 20
seq_len = 5
batch_size = 1

# Create a dummy input tensor
dummy_input = torch.randn(seq_len, batch_size, input_size)

# Run the RNN
rnn = SimpleRNN(input_size, hidden_size)
all_hidden_states = rnn(dummy_input)

# Visualize the final hidden state
final_hidden_state = all_hidden_states[-1].detach().numpy()

plt.figure(figsize=(10, 2))
plt.imshow(final_hidden_state, cmap='viridis', aspect='auto')
plt.title(f'Final Hidden State (The Bottleneck)\n(Shape: {final_hidden_state.shape})')
plt.xlabel('Hidden Dimension')
plt.yticks([])
plt.colorbar()
plt.show()

print("The plot above shows the final hidden state vector. It must contain a summary of the entire input sequence.")

## 🎯 Section 4: Core Research Content - Deconstructing the Transformer

### Scaled Dot-Product Attention

This is the fundamental building block. It takes three inputs: a **Query** (Q), a **Key** (K), and a **Value** (V). Imagine looking up a recipe: your query is "how to make bread." You scan the keys (chapter titles) in a cookbook. When you find a good match ("Artisan Breads"), you then retrieve the value (the actual recipe content).

- **Queries (Q)**: A matrix where each row is a vector representing a token's question about its context.
- **Keys (K)**: A matrix where each row is a vector representing a token's "advertisement" of what it contains.
- **Values (V)**: A matrix where each row is a vector representing the actual content of a token, to be aggregated.

In **self-attention**, Q, K, and V all originate from the same input sequence (though they are projected through different linear layers first).

#### The Formula
$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$

#### Step-by-Step Breakdown:
1.  **$QK^T$**: Compute the dot product between every query and every key. This results in a matrix of raw attention scores. The element at `(i, j)` is the score for how much query `i` should attend to key `j`.
2.  **$/ \sqrt{d_k}$**: Scale the scores. As the lecture notes, this is crucial for stabilizing training. When the dimension of the key vectors ($d_k$) is large, the dot products can grow very large in magnitude, pushing the softmax function into regions with tiny gradients. Scaling by $\sqrt{d_k}$ counteracts this.
3.  **softmax(...)**: Apply the softmax function to the scaled scores, converting them into a probability distribution (the attention weights). Each row now sums to 1.
4.  **$\dots V$**: Multiply the attention weights by the Value matrix. This creates a weighted sum of the value vectors, where the weights are determined by the attention scores. The output for each query is a new vector that is a blend of all the value vectors from the sequence, with more focus on the relevant ones.

In [ ]:
def scaled_dot_product_attention_educational(q, k, v, mask=None):
    """Educational implementation of Scaled Dot-Product Attention."""
    # q, k, v are of shape (..., seq_len, dim)
    d_k = q.shape[-1]
    print(f"--- Step-by-step Attention ---")
    print(f"Query/Key/Value dimension (d_k): {d_k}")
    
    # 1. MatMul: Q * K^T
    # Transpose the last two dimensions of K
    scores = torch.matmul(q, k.transpose(-2, -1))
    print(f"1. Scores (Q @ K.T) shape: {scores.shape}")
    
    # 2. Scale
    scaled_scores = scores / np.sqrt(d_k)
    print(f"2. Scaled scores shape: {scaled_scores.shape}")
    
    # 3. Mask (optional, for decoders)
    if mask is not None:
        # Set masked positions to a very large negative number
        scaled_scores = scaled_scores.masked_fill(mask == 0, -1e9)
        print(f"3. Applied mask.")
    
    # 4. Softmax
    attention_weights = F.softmax(scaled_scores, dim=-1)
    print(f"4. Attention weights (after softmax) shape: {attention_weights.shape}")

    # 5. MatMul with V
    output = torch.matmul(attention_weights, v)
    print(f"5. Final output (weights @ V) shape: {output.shape}")
    print("-------------------------------")
    
    return output, attention_weights

def scaled_dot_product_attention_optimized(q, k, v, mask=None):
    """Optimized implementation."""
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, v)
    return output, attention_weights

# --- Visualization ---
import torch.nn.functional as F
import seaborn as sns

seq_len = 5
d_k = 8 # Dimension of q, k, v
batch_size = 1

# Create dummy tensors
Q = torch.randn(batch_size, seq_len, d_k)
K = torch.randn(batch_size, seq_len, d_k)
V = torch.randn(batch_size, seq_len, d_k)

# Run the educational version
output, attention_weights = scaled_dot_product_attention_educational(Q, K, V)

# Visualize the attention weights
plt.figure(figsize=(6, 5))
sns.heatmap(attention_weights.squeeze(0).detach().numpy(), annot=True, cmap='viridis', fmt='.2f')
plt.title('Attention Weights Matrix')
plt.xlabel('Key Positions')
plt.ylabel('Query Positions')
plt.show()

print("The heatmap shows the attention weights. Row i shows the distribution of attention from query i to all keys.")
print(f"For example, the output for query 0 is a weighted sum of all 5 value vectors, using the weights from the first row: {np.round(attention_weights.squeeze(0)[0].detach().numpy(), 2)}")

### Multi-Head Attention

A single attention mechanism might be forced to average over different types of relationships (e.g., syntactic, positional, semantic). As the lecture points out, this can "mush together" information. Multi-Head Attention provides a solution by running the attention mechanism multiple times in parallel with different, learned linear projections.

#### Intuition
Think of it as giving the model multiple "points of view":
- **Head 1** might learn to focus on the next word in the sequence.
- **Head 2** might learn to connect verbs to their subjects, even if they are far apart.
- **Head 3** might learn to identify words that are semantically similar.

#### The Process
1.  **Project**: Take the input Q, K, and V and pass them through separate linear layers for each head. This creates `h` sets of `(Q_i, K_i, V_i)` for each head `i`.
2.  **Attend**: Apply scaled dot-product attention independently to each `(Q_i, K_i, V_i)` in parallel, producing `h` output vectors.
3.  **Concatenate**: Concatenate the `h` output vectors back together.
4.  **Final Projection**: Pass the concatenated result through a final linear layer to produce the final output, blending the information from all heads.

Crucially, the dimensions are adjusted so that the total computation is similar to a single attention head with the full model dimension, making it efficient.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear layers for Q, K, V and the final output
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
    def forward(self, q, k, v, mask=None):
        # q, k, v shape: (batch_size, seq_len, d_model)
        batch_size = q.size(0)
        
        # 1. Project and reshape for multi-head
        # (batch_size, seq_len, d_model) -> (batch_size, num_heads, seq_len, d_k)
        q = self.W_q(q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_k(k).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        v = self.W_v(v).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # 2. Apply attention on all heads in parallel
        x, attn_weights = scaled_dot_product_attention_optimized(q, k, v, mask)
        # x shape: (batch_size, num_heads, seq_len, d_k)
        # attn_weights shape: (batch_size, num_heads, seq_len, seq_len)
        
        # 3. Concatenate heads and apply final linear layer
        # (batch_size, num_heads, seq_len, d_k) -> (batch_size, seq_len, d_model)
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        output = self.W_o(x)
        
        return output, attn_weights

# --- Parameters ---
d_model = 64
num_heads = 4
seq_len = 10
batch_size = 1

# Create dummy input
input_tensor = torch.randn(batch_size, seq_len, d_model)

# Run MHA
mha = MultiHeadAttention(d_model, num_heads)
output, attn_weights = mha(input_tensor, input_tensor, input_tensor)

print(f"Input shape: {input_tensor.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")

# --- Visualize Attention from multiple heads ---
fig, axes = plt.subplots(1, num_heads, figsize=(20, 4))
for i in range(num_heads):
    sns.heatmap(attn_weights.squeeze(0)[i].detach().numpy(), ax=axes[i], cmap='cividis', cbar=False)
    axes[i].set_title(f'Attention Head {i+1}')
    axes[i].set_xlabel('Key Positions')
    if i == 0:
        axes[i].set_ylabel('Query Positions')
plt.suptitle('Attention Patterns from Different Heads')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

print("Each heatmap shows a different attention pattern. In a trained model, these would specialize to capture different relationships.")

### Positional Encodings

As the lecture states, self-attention is **permutation invariant**: if you shuffle the input words, the attention outputs (before adding them back to the original input) would just be shuffled in the same way. The model has no inherent sense of word order. This is a problem, as word order is critical to meaning ("the cat chased the dog" vs. "the dog chased the cat").

The original Transformer paper solved this by adding a **Positional Encoding** (PE) vector to each input embedding. This vector is not learned; it's generated by a clever formula using sine and cosine functions of different frequencies.

#### The Formula
For a token at position `pos` and dimension `i` in the embedding, the encoding is:
$$ PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right) $$
$$ PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right) $$

#### Intuition
The key idea is that for any fixed offset `k`, $PE_{pos+k}$ can be represented as a linear function of $PE_{pos}$. This means the model can easily learn to attend to relative positions. Each dimension of the encoding corresponds to a sinusoid of a different wavelength, from very long to very short, allowing the model to uniquely identify positions and their relationships.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add a batch dimension so we can easily add it to the input embeddings
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        # Add the positional encoding to the input
        # self.pe is of shape (1, max_len, d_model)
        # We slice it to the sequence length of x
        return x + self.pe[:, :x.size(1), :]

# --- Visualize the Positional Encoding Matrix ---
d_model = 128
max_len = 100

pe_layer = PositionalEncoding(d_model, max_len)
positional_encodings = pe_layer.pe.squeeze(0).numpy()

plt.figure(figsize=(10, 6))
plt.pcolormesh(positional_encodings, cmap='RdBu')
plt.xlabel('Embedding Dimension')
plt.xlim((0, d_model))
plt.ylabel('Position in Sequence')
plt.ylim((max_len, 0))
plt.colorbar()
plt.title('Sinusoidal Positional Encodings')
plt.show()

print("Each row is the unique positional vector for a token at that position.")
print("The columns vary from low-frequency (left) to high-frequency (right) sinusoids.")

### Assembling the Blocks: Encoder and Decoder Layers

A full Transformer is built by stacking these core components into **Encoder** and **Decoder** layers.

#### The Encoder Layer
An encoder layer has two main sub-layers:
1.  **Multi-Head Self-Attention**: The mechanism we just discussed.
2.  **Position-wise Feed-Forward Network (FFN)**: A simple two-layer fully connected network applied independently to each position.

Crucially, each of these sub-layers has a **residual connection** around it, followed by **layer normalization**. This is a key technique for enabling stable training of very deep networks.
$$ \text{SublayerOutput} = \text{LayerNorm}(x + \text{Sublayer}(x)) $$

#### The Decoder Layer
A decoder layer is similar but has a third sub-layer:
1.  **Masked Multi-Head Self-Attention**: Same as the encoder's self-attention, but with a "look-ahead mask". This prevents a position from attending to subsequent positions. This is essential during training, as the model should only use past ground-truth words to predict the next word, simulating how it would behave at inference time.
2.  **Encoder-Decoder Attention**: Here, the **Queries (Q)** come from the output of the previous decoder sub-layer, while the **Keys (K) and Values (V)** come from the **final output of the entire encoder stack**. This is the step where the decoder consults the encoded representation of the input sentence to generate the output.
3.  **Position-wise Feed-Forward Network**: Identical to the one in the encoder.

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src, src_mask):
        # Attention sub-layer
        attn_output, _ = self.self_attn(src, src, src, src_mask)
        src = self.norm1(src + self.dropout(attn_output))
        
        # Feed-forward sub-layer
        ff_output = self.feed_forward(src)
        src = self.norm2(src + self.dropout(ff_output))
        
        return src

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, tgt, memory, tgt_mask, memory_mask):
        # Masked self-attention sub-layer
        attn_output, _ = self.self_attn(tgt, tgt, tgt, tgt_mask)
        tgt = self.norm1(tgt + self.dropout(attn_output))
        
        # Encoder-decoder attention sub-layer
        # Q from decoder, K and V from encoder memory
        enc_dec_attn_output, _ = self.enc_dec_attn(tgt, memory, memory, memory_mask)
        tgt = self.norm2(tgt + self.dropout(enc_dec_attn_output))
        
        # Feed-forward sub-layer
        ff_output = self.feed_forward(tgt)
        tgt = self.norm3(tgt + self.dropout(ff_output))
        
        return tgt

print("EncoderLayer and DecoderLayer classes defined.")
print("Notice the 'Add & Norm' pattern: `norm(x + dropout(sublayer(x)))`")

## 🧪 Section 5: Experimental Analysis & Intuition Building

### Experiment 1: The Importance of the Scaling Factor

The lecture mentions that the scaling factor $\sqrt{d_k}$ is critical for stable training. Let's verify this. We'll compare the output distribution of the softmax function with and without scaling as the key dimension ($d_k$) increases.

**Hypothesis**: Without scaling, as $d_k$ increases, the dot products will have a larger variance. This will push the softmax inputs to be very large or very small, resulting in an extremely "spiky" (one-hot) distribution where the gradient for non-maximal values is almost zero, halting learning.

In [ ]:
@widgets.interact(
    d_k=widgets.IntSlider(value=8, min=2, max=256, step=2, description='d_k'),
    scale=widgets.Checkbox(value=True, description='Use Scaling?')
)
def scaling_factor_explorer(d_k, scale):
    # Assume Q and K have elements drawn from a standard normal distribution
    q = torch.randn(1, d_k)
    k = torch.randn(10, d_k) # 10 keys to attend to
    
    scores = torch.matmul(q, k.T).squeeze()
    
    if scale:
        scaled_scores = scores / np.sqrt(d_k)
        title = f'With Scaling (d_k={d_k})'
        data_to_plot = scaled_scores
    else:
        title = f'Without Scaling (d_k={d_k})'
        data_to_plot = scores
    
    probs = F.softmax(data_to_plot, dim=-1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.bar(range(10), data_to_plot.detach().numpy(), color='skyblue')
    ax1.set_title(f'Scores before Softmax\nVariance: {data_to_plot.var():.2f}')
    ax1.set_xlabel('Key Position')
    
    ax2.bar(range(10), probs.detach().numpy(), color='salmon')
    ax2.set_title(f'Probabilities after Softmax\n{title}')
    ax2.set_xlabel('Key Position')
    ax2.set_ylim(0, 1)
    
    plt.tight_layout()
    plt.show()
    
    print("Instructions: Uncheck 'Use Scaling' and increase d_k. Notice how the score variance grows and the probability distribution collapses to a single point. Now re-check 'Use Scaling'; the variance stays around 1.0, and the distribution is much softer, allowing for richer gradients.")

### Experiment 2: Training a Toy Model to See Why Positional Encodings are Necessary

Let's create a very simple sequence-to-sequence task: reversing a sequence of numbers. A Transformer should be able to solve this easily. We will train two small models:
1.  A standard Transformer with positional encodings.
2.  An identical Transformer but with positional encodings **removed**.

**Hypothesis**: The model with positional encodings will learn the task perfectly. The model without them will fail completely, as it has no way to know the order of the numbers it needs to reverse.

In [ ]:
# A simplified full Transformer model for this toy task
# (Full code omitted for brevity in this example cell, assuming it's defined)
class ToyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, use_pe=True):
        super().__init__()
        self.use_pe = use_pe
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layers = [EncoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)]
        self.encoder = nn.Sequential(*encoder_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)
    
    def forward(self, src):
        embedded = self.embedding(src)
        if self.use_pe:
            x = self.pos_encoder(embedded)
        else:
            x = embedded
        encoded = self.encoder(x)
        return self.fc_out(encoded)

def train_reversal_task(use_pe=True):
    # Simple training loop for the reversal task
    vocab_size = 20
    seq_len = 10
    model = ToyTransformer(vocab_size=vocab_size, d_model=32, num_heads=2, num_layers=1, d_ff=64, use_pe=use_pe)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    losses = []
    
    print(f"--- Training {'with' if use_pe else 'without'} Positional Encodings ---")
    for epoch in range(200):
        # Generate a batch of data
        src = torch.randint(1, vocab_size, (16, seq_len)) # 0 is padding
        tgt = torch.flip(src, dims=[1])
        
        optimizer.zero_grad()
        output = model(src)
        # Reshape for loss function
        loss = criterion(output.view(-1, vocab_size), tgt.view(-1))
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    print("Training complete.")
    return losses, model

# Train both models
losses_with_pe, model_with_pe = train_reversal_task(use_pe=True)
losses_without_pe, model_without_pe = train_reversal_task(use_pe=False)

# Plot the losses
plt.figure(figsize=(10, 5))
plt.plot(losses_with_pe, label='With Positional Encodings')
plt.plot(losses_without_pe, label='Without Positional Encodings')
plt.title('Training Loss for Sequence Reversal Task')
plt.xlabel('Training Step')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# Test inference
test_seq = torch.tensor([[5, 8, 2, 10, 4, 1, 1, 1, 1, 1]])
expected = torch.flip(test_seq, dims=[1])

with torch.no_grad():
    pred_with_pe = model_with_pe(test_seq).argmax(dim=-1)
    pred_without_pe = model_without_pe(test_seq).argmax(dim=-1)

print("--- INFERENCE TEST ---")
print(f"Input Sequence:          {test_seq.numpy()}")
print(f"Expected Reversed Seq:     {expected.numpy()}")
print(f"Prediction WITH PE:        {pred_with_pe.numpy()}")
print(f"Prediction WITHOUT PE:     {pred_without_pe.numpy()}")
print("\nResult: The model with positional encodings learns to reverse the sequence, while the other fails, confirming our hypothesis.")

## 🚀 Section 6: Advanced Topics & Research Extensions

### Efficient Attention: The Problem of $O(n^2)$ Complexity

The biggest weakness of the vanilla Transformer is the self-attention computation. To compute the attention scores, we perform a matrix multiplication between Q (shape `n x d`) and K transposed (shape `d x n`), resulting in an `n x n` attention matrix, where `n` is the sequence length. This $O(n^2)$ complexity in both time and memory makes it prohibitively expensive to use Transformers on very long sequences (e.g., entire books, high-resolution images).

#### Conceptual Idea behind FlashAttention

The lecture mentions FlashAttention as a key system-level optimization. It doesn't change the mathematical result of attention, but it changes *how* it's computed on the GPU. 

A standard implementation is **memory-bound**. It involves multiple separate GPU kernel calls:
1. Compute $S = QK^T$. Write the huge $n \times n$ matrix $S$ from fast SRAM to slow HBM (GPU main memory).
2. Read $S$ from HBM back to SRAM. Compute $P = \text{softmax}(S)$. Write $P$ back to HBM.
3. Read $P$ and $V$ from HBM. Compute $O = PV$. Write $O$ back to HBM.

The bottleneck is not the math (FLOPs), but the constant reading and writing to slow HBM memory.

**FlashAttention fuses these operations into a single GPU kernel.** It computes the attention matrix in smaller blocks (tiling), and performs the softmax and multiplication with V *within the fast SRAM* without ever writing the full $n \times n$ attention matrix to HBM. This dramatically speeds up the process and reduces memory usage, enabling models to handle much longer contexts.

### Improved Positional Information: Rotary Position Embedding (RoPE)

The lecture notes that a major improvement over the original sinusoidal encodings are **Rotary Position Embeddings (RoPE)**. While the original PE *adds* positional information, RoPE *rotates* the query and key vectors based on their absolute position.

**Core Idea**: The dot product between two vectors, $q$ and $k$, depends on their relative angle. RoPE applies a rotation matrix to the query and key vectors that depends on their absolute position ($m$ and $n$). When you compute the dot product of the rotated vectors, $R_m q$ and $R_n k$, the final result magically only depends on the vectors themselves and their *relative position* ($m-n$).

This is elegant because it injects relative position information directly into the self-attention mechanism itself, rather than adding it beforehand. It has shown superior extrapolation capabilities to longer sequences than the original PE.

## 💼 Section 7: Practical Applications & The Big Picture

The lecture concludes by emphasizing the incredible consolidation the Transformer has brought to the field of AI. The same fundamental architecture, with minor modifications, now powers a vast range of applications, bringing us closer to the Dartmouth Conference's original vision.

### Key Applications:
- **Large Language Models (LLMs)**: Models like GPT, LLaMA, and Claude are typically decoder-only Transformers trained on massive web-scale text to perform next-token prediction. Their emergent abilities in reasoning, translation, and code generation stem from this simple objective combined with unprecedented scale.
- **Machine Translation**: The original task for the Transformer, where an encoder processes the source language and a decoder generates the target language.
- **Summarization**: An encoder-decoder model can read a long document and generate a concise summary.
- **Computer Vision (Vision Transformer - ViT)**: An image is broken down into a sequence of patches, which are then treated as "tokens" and fed into a standard Transformer encoder for classification.
- **Music and Speech (Music Transformer, Whisper)**: Audio is tokenized into a sequence that a Transformer can process for generation, transcription, or understanding.

### The Future: A General Purpose Computer

The lecture suggests we are moving towards a future where LLMs act as a new kind of general-purpose computer. Instead of writing code, we write prompts. The model's ability to use tools, interact with APIs, and orchestrate other agents points to a new paradigm of human-computer interaction.

The journey from disparate, rule-based systems to a single, scalable, and generalizable architecture like the Transformer is a testament to the power of finding the right foundational principles—in this case, that **attention is all you need**.